In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/retail_sales_dataset.csv')
print(f"Raw dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

Raw dataset: 120,000 rows × 17 columns


First we load the raw dataset. Here all transformations are applied and the result is saved as a separated clean file.

In [3]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

print(f"Type before: object → after: {df['transaction_date'].dtype}")
print(f"Range: {df['transaction_date'].min().date()} → {df['transaction_date'].max().date()}")

Type before: object → after: datetime64[ns]
Range: 2024-01-01 → 2025-12-30


Convert transaction_date to datatime, this enables time-series operations, grouping by period, and feature extration.

In [5]:
df['year']        = df['transaction_date'].dt.year
df['month']       = df['transaction_date'].dt.month
df['month_name']  = df['transaction_date'].dt.strftime('%B')
df['quarter']     = df['transaction_date'].dt.quarter
df['day_of_week'] = df['transaction_date'].dt.strftime('%A')
df['is_weekend']  = df['transaction_date'].dt.dayofweek >= 5

print(df[['transaction_date','year','month','month_name',
          'quarter','day_of_week','is_weekend']].head())

  transaction_date  year  month month_name  quarter day_of_week  is_weekend
0       2024-04-24  2024      4      April        2   Wednesday       False
1       2025-07-12  2025      7       July        3    Saturday        True
2       2025-06-01  2025      6       June        2      Sunday        True
3       2025-08-26  2025      8     August        3     Tuesday       False
4       2024-12-10  2024     12   December        4     Tuesday       False


From the date column six new features are extracted. These enable to make a dashboard that answers questions like: which month generates the most revenue, do weekends drive more sales, and how does performance compare quarter over quarter.

In [6]:
def classify_sale(amount):
    if amount < 100:
        return 'Small'
    elif amount < 500:
        return 'Medium'
    elif amount < 1000:
        return 'Large'
    else:
        return 'Premium'

df['sales_tier'] = df['sales_amount'].apply(classify_sale)

print(df['sales_tier'].value_counts())
print(f"\nPremium sales %: {(df['sales_tier'] == 'Premium').mean()*100:.1f}%")

sales_tier
Medium     71334
Small      23119
Large      18572
Premium     6975
Name: count, dtype: int64

Premium sales %: 5.8%


Classifications of each sale into four tiers based on sales amount. This helps the dashboard to segment revenue by transaction size, and adds context to the sales_amount outliers found in the EDA. They're valid premium purchases, not data errors.

In [7]:
df['revenue_per_unit'] = (df['sales_amount'] / df['quantity']).round(2)

print(df[['product_name','quantity','sales_amount','revenue_per_unit']].head(10))

    product_name  quantity  sales_amount  revenue_per_unit
0      Dumbbells         2        501.65            250.82
1  Running Shoes         1        366.16            366.16
2       Sneakers         1         27.99             27.99
3      Sunscreen         2        173.42             86.71
4       Sneakers         1        259.55            259.55
5          Bread         1        474.70            474.70
6           Rice         1         53.35             53.35
7     Board Game         4         70.32             17.58
8          Jeans         1        277.08            277.08
9   Cookware Set         1         31.42             31.42


Having the revenue per unit normalizes sales across transactions with different quantities, this makes products commparitions fairer.

In [8]:
df['has_discount'] = df['discount_pct'] > 0

discount_rate = df['has_discount'].mean() * 100
print(f"Transactions with a discount applied: {discount_rate:.1f}%")
print(df['has_discount'].value_counts())

Transactions with a discount applied: 40.0%
has_discount
False    71956
True     48044
Name: count, dtype: int64


This flag makes it easier to compare discounted vs full-price transactions in the dashboard. This also confirms the EDA finding that about 50% of transactions have no discount applied.

In [9]:
print("=== Final dataset ===")
print(f"Shape: {df.shape}")
print(f"\nNew columns added: {df.shape[1] - 17}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nNull check:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

=== Final dataset ===
Shape: (120000, 26)

New columns added: 9

Dtypes:
transaction_id                object
transaction_date      datetime64[ns]
customer_id                   object
customer_gender               object
customer_age_group            object
customer_segment              object
product_id                    object
product_name                  object
category                      object
brand                         object
quantity                       int64
unit_price                   float64
discount_pct                   int64
sales_amount                 float64
payment_method                object
sales_channel                 object
region                        object
year                           int32
month                          int32
month_name                    object
quarter                        int32
day_of_week                   object
is_weekend                      bool
sales_tier                    object
revenue_per_unit             float64
ha

In [10]:
df.to_csv('../data/clean/retail_sales_dataset_clean.csv', index=False)
print("Clean dataset saved to data/clean/retail_sales_clean.csv")
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Clean dataset saved to data/clean/retail_sales_clean.csv
Final shape: 120,000 rows × 26 columns


Transformation summary:

|Transformation|Type|Colums added|Reason|
|---|---|---|---|
|Data type conversion|Fix|None|Enable datetime operations|
|year/month/quarter|Feature engineering|3|Time-series grouping|
|Month name/day of week|Feature engineering|2|Readable lables for dashboard|
|Is weekend|Feature engineering|1|Behavioral pattern analysis|
|Sales tier|Feature engineering|1|Segment transactions by size|
|Revenue per unit|Feature engineering|1|Normalized product comparison|
|Has discount|Feature engineering|1|Filter discounted vs full-price|